In [ ]:
# Parameters -- Fabric overrides at runtime.
tenant_id   = "100"        # the practice to freeze/restore
direction   = "freeze"     # "freeze" (stage_* -> init_stage_*) or "restore" (init_stage_* -> stage_*)
init_prefix = "init_"      # frozen tables live as init_stage_<entity> in the SAME lakehouse


In [ ]:
tid = str(tenant_id)
where = "tenant_id = '" + tid + "'"

def copy_one(src_tbl, dst_tbl, stamp=None):
    df = spark.table(src_tbl).where(where)
    n = df.count()
    if n == 0:
        print("  skip " + src_tbl + " (0 rows for tenant " + tid + ")")
        return 0
    if stamp is not None:                        # restore: re-stamp so onboarding's sync-poll sees it
        from pyspark.sql.functions import lit
        df = df.withColumn("DW_Stage_Loaded_At", lit(stamp))
    w = df.write.format("delta")
    if spark.catalog.tableExists(dst_tbl):
        w = w.mode("overwrite").option("replaceWhere", where).option("mergeSchema", "true")
    else:
        w = w.mode("overwrite").option("overwriteSchema", "true")
    w.saveAsTable(dst_tbl)
    print("  " + src_tbl + " -> " + dst_tbl + "  (" + str(n) + " rows)")
    return n

all_tables = [t.name for t in spark.catalog.listTables()]

if direction == "freeze":
    # every stage_* table (NOT the init_stage_* ones) -> init_stage_*
    srcs = [t for t in all_tables if t.startswith("stage_") and not t.startswith(init_prefix)]
    print("FREEZE tenant " + tid + ": " + str(len(srcs)) + " stage tables -> " + init_prefix + "stage_*")
    total = sum(copy_one(t, init_prefix + t) for t in srcs)
    print("Frozen " + str(total) + " rows. init_stage_* is now the onboarding snapshot; deltas may overwrite stage_* safely.")
elif direction == "restore":
    # every init_stage_* table -> the matching stage_* (strip the init_ prefix)
    srcs = [t for t in all_tables if t.startswith(init_prefix + "stage_")]
    print("RESTORE tenant " + tid + ": " + str(len(srcs)) + " frozen tables -> stage_*")
    total = sum(copy_one(t, t[len(init_prefix):]) for t in srcs)
    print("Restored " + str(total) + " rows into stage_*. Now rebuild Bronze->Gold (build-only).")
else:
    raise SystemExit("direction must be 'freeze' or 'restore', got: " + repr(direction))
